¿Qué debe de tener este notebook?
1. Diario de laboratorio donde documentes las ejecuciones de Weights & Biases (W&B), los recortes de hiperparámetros (la "golden recipe" frente a la búsqueda en cuadrícula) y el análisis de la memoria VRAM.
2. Arnés de Evaluación (Evaluation Harness): no vamos a construir el harness desde cero. Esta evaluación se ejecuta desde la terminal, totalmente fuera de tu script sft_trainer.py. Evaluación con bigcode-evaluation-harness (HumanEval/MBPP) en vez de construir el harness desde cero.

"""
FALTA: Evaluación con bigcode-evaluation-harness (HumanEval/MBPP) en vez de construir el harness desde cero.
Medir empíricamente si tu SFT ha mejorado la capacidad de codificación del modelo base usando un arnés estandarizado.Qué tienes que hacer:Esto no va en tu script de Python actual. Deberás clonar el repositorio independiente de bigcode-evaluation-harness en tu entorno.  Una vez tu modelo termine de entrenar y se guarde el checkpoint, utilizarás la CLI del harness apuntando a la ruta de tu adaptador y al modelo base para ejecutar los benchmarks de HumanEval o MBPP.  
Evaluación con bigcode-evaluation-harness (HumanEval/MBPP) en vez de construir el harness desde cero.
Checkpoint model-sft-checkpoint.
"""

Training Decisions o Compute Budget Justification.
"Se descartó la optimización de hiperparámetros bayesiana (Optuna) porque el presupuesto computacional (1x GPU 24GB) hace inviable iterar múltiples SFTs completos de un modelo 7B. En su lugar, se adoptaron hiperparámetros LoRA estándar de la industria, reservando el cómputo para la generación de pares DPO."

TRADUCIR AL INGLÉS Y REFINAR !!!

# Fase 2: Supervised Fine-Tuning (SFT) y Evaluación Baseline

## 1. Training Decisions & Compute Budget Justification

Se descartó la optimización de hiperparámetros bayesiana (Optuna) porque el presupuesto computacional (1x GPU 24GB) hace inviable iterar múltiples SFTs completos de un modelo 7B. En su lugar, se adoptaron hiperparámetros LoRA estándar de la industria ("Golden Recipe"), reservando el cómputo para la generación de pares DPO y la fase final de alineación mediante recompensa compuesta.

**Hiperparámetros adoptados (Golden Recipe):**
* `r = 16`, `lora_alpha = 32`, `dropout = 0.1`
* `learning_rate = 2e-4` con optimizador `paged_adamw_8bit`
* Precisión mixta: `bfloat16` con base cuantizada en `NF4` (QLoRA).

## 2. Diario de Laboratorio: W&B y Análisis de Memoria

### Consumo de VRAM
Para encajar el modelo Qwen2.5-Coder-7B en una tarjeta de 24GB, aplicamos las siguientes técnicas de mitigación de memoria:
1. **QLoRA (4-bit):** Reduce el modelo base de ~14GB (en fp16) a ~4.5GB.
2. **Gradient Checkpointing:** Intercambia cómputo extra por ahorro masivo de memoria durante el pase hacia atrás (backward pass).
3. **Batch Size y Accumulation:** Se utilizó un `per_device_train_batch_size=2` con `gradient_accumulation_steps=8`, logrando un batch size efectivo de 16 sin picos de OOM (Out Of Memory).

*Consumo estabilizado:* ~16-18 GB VRAM.

### Weights & Biases Logging
*(Nota para ti: Cuando ejecutes tu entrenamiento, saca una captura de pantalla de las gráficas de `train/loss` y `learning_rate` de W&B y arrástralas aquí, o pon el enlace público a tu reporte).*

- **Enlace al reporte de W&B:** [Añadir link aquí]
- **Observaciones:** La pérdida (loss) convergió de manera estable sin picos anómalos...

## 3. Arnés de Evaluación (Evaluation Harness)

Para medir empíricamente si el SFT ha mejorado la capacidad de codificación del modelo base, utilizaremos el arnés estandarizado `bigcode-evaluation-harness`. Esta evaluación se ejecuta por terminal, aislando por completo la validación del script de entrenamiento (`sft_trainer.py`).

In [ ]:
!git clone https://github.com/bigcode-project/bigcode-evaluation-harness.git tools/bigcode-evaluation-harness
!cd tools/bigcode-evaluation-harness && pip install -e .
!mkdir -p data/evaluation

In [ ]:
!accelerate launch tools/bigcode-evaluation-harness/main.py \
  --model Qwen/Qwen2.5-Coder-7B-Instruct \
  --tasks humaneval \
  --precision bf16 \
  --allow_code_execution \
  --save_generations \
  --save_generations_path data/evaluation/baseline_generations.json \
  --metric_output_path data/evaluation/baseline_metrics.json \
  --limit 20 # IMPORTANTE: Quita el --limit 20 en la ejecución final. Se usa aquí para testear que el arnés funciona rápido.

In [ ]:
import json

def load_metrics(filepath):
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return None

base_metrics = load_metrics('data/evaluation/baseline_metrics.json')
sft_metrics = load_metrics('data/evaluation/sft_metrics.json')

if base_metrics and sft_metrics:
    base_pass = base_metrics.get('humaneval', {}).get('pass@1', 0) * 100
    sft_pass = sft_metrics.get('humaneval', {}).get('pass@1', 0) * 100
    
    print(f"--- Results HumanEval (Pass@1) ---")
    print(f"Base Model: {base_pass:.2f}%")
    print(f"SFT Model: {sft_pass:.2f}%")
    print(f"Upgrade: {sft_pass - base_pass:+.2f} points")
else:
    print("Metrics files are not generated yet. Execute the above cells.")